In [0]:
!pip install yfinance -q

In [0]:
import yfinance as yf
import pandas as pd

In [0]:
tickers = 'BTC-USD'
periodo = '1y'
intervalo = '1d'

dados_bronze = yf.download(tickers, period=periodo, interval=intervalo)
dados_bronze = dados_bronze.reset_index()
dados_bronze.columns = [coluna[0] for coluna in dados_bronze.columns]

dados_bronze.rename(columns={
    'Date': 'data', 
    'Close': 'fechamento', 
    'High': 'maximo', 
    'Low': 'minimo', 
    'Open': 'abertura', 
    'Volume': 'volume'}, inplace=True)
display(dados_bronze.head())

In [0]:
dados_bronze.dtypes

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark

In [0]:
df_spark = spark.createDataFrame(dados_bronze)

df_spark = (
    df_spark
    .withColumn("data", df_spark["data"].cast("date"))
    .withColumn("volume", df_spark["volume"].cast("double"))
)

(
    df_spark
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("db_analytics.bronze_bitcoin")
)

print(df_spark.count())